# 05 Hanapin ang pinagmulan — notebook

Ipinakita ng kabanata 04 ang *kailan*, *saan* at *sino*. Ngayon, tine-test ni Maya ang
**bakit**: aling mga exposure pagkatapos ng Bagyong Amihan ang nagpataas ng tsansa ng
leptospirosis? Ininterbyu ng team ni Doc Mon ang **120 kaso at 240 control** (dalawang
kapitbahay mula sa parehong barangay para sa bawat kaso) sa walong barangay sa
tabing-ilog. Synthetic ang study, ang mga tao at ang bawat numero.

Gagawa ka ng 2×2 tables gamit ang `pd.crosstab`, kukuwentahin nang mano-mano ang odds
ratio at 95% confidence interval, iche-check ang mga ito laban sa scipy at statsmodels,
huhuli ng nabaligtad na table, at aayusin ang isang confounder gamit ang Mantel–Haenszel
method.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## Hakbang 1 · I-load ang study at tingnan

Bakit **case-control study**? Hindi karaniwan ang leptospirosis kahit may baha, kaya kung
susundan ang buong lungsod (isang *cohort*), kailangang interbyuhin ang sampu-sampung
libong tao para makahanap ng ilang daang kaso. Sa halip, nagsisimula tayo sa mga **kaso**
na mayroon na tayo, pumipili ng mga **control** na hindi nagkasakit mula sa parehong mga
barangay, at ikinukumpara ang ginawa ng dalawang grupo pagkatapos ng bagyo.

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
from scipy.stats.contingency import odds_ratio

cc = pd.read_csv("data/synthetic/ph/lepto_case_control.csv")
print(len(cc), "kalahok:", cc["status"].value_counts().to_dict())
print("Matched sets:", cc["matched_set"].nunique(), "| barangays:", ", ".join(sorted(cc["barangay"].unique())))
cc.head(6)

## Hakbang 2 · Nagkamali si Tala: ang baligtad na table

> **Tanungin si Tala:** "Sa `cc`, 0/1 ang `waded_floodwater` at `case`/`control` ang
> `status`. Gawin ang 2×2 table at i-compute ang odds ratio ng paglusong sa baha."

Binubuo ni Tala ang table at binabasa ang apat na cell **ayon sa posisyon**: *a* ang nasa
kaliwang itaas, *b* ang nasa kanang itaas, at iba pa.

In [ ]:
# Unang sagot ni Tala -- binasa ang mga cell ayon sa POSISYON, hindi tiningnan ang mga label
t = pd.crosstab(cc["waded_floodwater"], cc["status"])
a, b = t.iloc[0, 0], t.iloc[0, 1]
c, d = t.iloc[1, 0], t.iloc[1, 1]
print(t)
print("OR ni Tala =", round(a * d / (b * c), 2))

Ang OR na 0.18 ay mangangahulugang **pinrotektahan** ng paglusong ang mga tao. Matatawa si
Aling Nena. Paano natin ito mahuhuli nang hindi alam ang sagot?

1. **Basahin ang mga label.** Inaayos ng `crosstab` ang rows na 0 bago 1, kaya ang *unang*
   row ay "**hindi** lumusong". Ang *a* ni Tala ay mga kasong hindi exposed, hindi mga
   kasong exposed. Isang pagkakasunod ang inakala ng formula; iba ang nasa table.
2. **I-sanity-check ang direksiyon.** Bago ang anumang formula, ikumpara ang simpleng
   porsiyento: mas marami bang kaso kaysa control ang lumusong? Kung oo, dapat lampas sa 1
   ang OR.

(Hindi magbabago ang OR kung pagpapalitin ang rows *at* columns — isang buong transpose.
Kapag isa lang ang binaligtad, ang **reciprocal** ang lalabas: 0.18 = 1 / 5.57.)

In [ ]:
waded_pct = pd.crosstab(cc["status"], cc["waded_floodwater"], normalize="index") * 100
print("Porsiyento ng lumusong:")
print(waded_pct[1].round(1))

Tatlo sa apat na kaso ang lumusong, laban sa mga isa sa tatlong control: dapat malayong
lampas sa 1 ang OR. Ngayon, ang ligtas na paraan. Inilalagay natin ang rows at columns sa
**nakapirmi at may-label na pagkakasunod** gamit ang `.loc[[1, 0], [1, 0]]` at
pinapangalanan ang mga ito, para mabasa ng kahit sino ang table:

|  | Cases | Controls |
|---|---|---|
| **Exposed** | a | b |
| **Hindi exposed** | c | d |

$$OR = \frac{a \times d}{b \times c} \qquad
95\%\ CI = \exp\left(\ln OR \pm 1.96\sqrt{\tfrac1a+\tfrac1b+\tfrac1c+\tfrac1d}\right)$$

Ang confidence interval na ito ay ang *Woolf* (log) method.

In [ ]:
def two_by_two(data: pd.DataFrame, exposure: str) -> pd.DataFrame:
    """2x2 table: rows exposed / hindi exposed, columns cases / controls."""
    table = pd.crosstab(data[exposure], data["case"]).loc[[1, 0], [1, 0]]
    table.index = ["Exposed", "Hindi exposed"]
    table.columns = ["Cases", "Controls"]
    return table


table = two_by_two(cc, "waded_floodwater")
(a, b), (c, d) = table.to_numpy()
or_wading = a * d / (b * c)
se = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
low, high = math.exp(math.log(or_wading) - 1.96 * se), math.exp(math.log(or_wading) + 1.96 * se)

print(table, "\n")
print(f"a={a} b={b} c={c} d={d}")
print(f"OR = {or_wading:.2f} (95% CI {low:.2f}–{high:.2f})")

**I-check natin** gamit ang dalawang magkahiwalay na library. Kailangang sabihan ang `odds_ratio`
ng scipy ng `kind="sample"` (ibang estimator ang default nito, ang *conditional* OR, na
bahagyang ibang numero ang ibinibigay). Pareho ang Woolf interval na gamit ng `Table2x2`
ng statsmodels.

In [ ]:
scipy_result = odds_ratio(table.to_numpy(), kind="sample")
scipy_ci = scipy_result.confidence_interval(confidence_level=0.95)
sm_table = sm.stats.Table2x2(table.to_numpy())
sm_low, sm_high = sm_table.oddsratio_confint()

print(f"mano-mano   : {or_wading:.2f} ({low:.2f}–{high:.2f})")
print(f"scipy       : {scipy_result.statistic:.2f} ({scipy_ci.low:.2f}–{scipy_ci.high:.2f})")
print(f"statsmodels : {sm_table.oddsratio:.2f} ({sm_low:.2f}–{sm_high:.2f})")
print(f"scipy default (conditional): {odds_ratio(table.to_numpy()).statistic:.2f}")

assert math.isclose(scipy_result.statistic, or_wading) and math.isclose(sm_table.oddsratio, or_wading)
assert math.isclose(scipy_ci.low, low, rel_tol=1e-3) and math.isclose(sm_high, high, rel_tol=1e-3)

## Hakbang 3 · Risk ratio o odds ratio?

Ikinukumpara ng **risk ratio** (RR) ang *risk* ng sakit sa mga exposed sa risk sa mga
hindi exposed: $\frac{a/(a+b)}{c/(c+d)}$. Kailangan diyan ang lahat ng at risk, gaya sa
cohort. Sa case-control study, **tayo ang pumili** kung ilang control ang kukunin, kaya
gawa ng ating design ang "risk" sa table natin. Tingnan ang mangyayari kung **isang**
control lang bawat kaso ang kinuha natin imbes na dalawa: gumagalaw ang "RR", halos hindi
ang OR.

In [ ]:
def fake_rr(t: pd.DataFrame) -> float:
    (a, b), (c, d) = t.to_numpy()
    return (a / (a + b)) / (c / (c + d))


one_control = cc[(cc["case"] == 1) | ~cc.duplicated(["matched_set", "case"])]  # unang control lang
for label, data in [("2 control bawat kaso", cc), ("1 control bawat kaso", one_control)]:
    t = two_by_two(data, "waded_floodwater")
    print(f"{label}: n = {len(data)} | OR = {sm.stats.Table2x2(t.to_numpy()).oddsratio:.2f}"
          f" | 'RR' = {fake_rr(t):.2f}  <- hindi totoong risk ratio")

Kaya: **cohort → RR; case-control → OR.** Kapag bihira ang sakit sa populasyon, malapit
ang OR sa RR, kaya puwede pa rin nating sabihing "mga limang beses ang risk" nang may
pag-iingat.

## Hakbang 4 · Bota sa mga lumusong, at doxycycline

Mahalaga lang ang bota sa mga taong **nasa** tubig, kaya sa mga lumusong lang itinanong
ang tungkol sa bota (blangko ang `wore_boots` para sa lahat ng iba). Kaya mga lumusong
lang muna ang kukunin natin. Ibinigay ang doxycycline prophylaxis sa mga evacuation center
at sa mga clean-up volunteer. Ang OR na **mas mababa sa 1** ay nangangahulugang mas
bihira ang exposure sa mga kaso: isang protective factor.

In [ ]:
waders = cc[cc["waded_floodwater"] == 1]
for label, data, exposure in [("Nakabota (mga lumusong lang)", waders, "wore_boots"),
                              ("Doxycycline prophylaxis", cc, "doxy_prophylaxis")]:
    t = two_by_two(data, exposure)
    result = sm.stats.Table2x2(t.to_numpy())
    lo, hi = result.oddsratio_confint()
    print(f"{label}: n = {len(data)}")
    print(t)
    print(f"OR = {result.oddsratio:.2f} (95% CI {lo:.2f}–{hi:.2f})\n")

## Hakbang 5 · Lahat ng exposure sa iisang larawan

> **Tanungin si Tala:** "Gamit ang function kong `two_by_two(data, exposure)` (rows
> exposed/hindi exposed, columns cases/controls), gumawa ng table na may a, b, c, d, OR at
> 95% CI para sa ilang 0/1 na column, saka gumuhit ng forest plot na may log x-axis at
> linya sa OR = 1."

Magandang *null* exposure ang `rats_near_home`: mahalaga ang daga sa leptospirosis sa
pangkalahatan, pero sa study na ito, hindi nakapagpaiba sa mga kaso at control ang
pagkakita ng daga malapit sa bahay.

In [ ]:
cc["male"] = (cc["sex"] == "M").astype(int)
EXPOSURES = [("waded_floodwater", cc), ("wore_boots", waders), ("doxy_prophylaxis", cc),
             ("skin_wound", cc), ("cleanup_work", cc), ("male", cc), ("rats_near_home", cc)]

rows = []
for exposure, data in EXPOSURES:
    t = two_by_two(data, exposure)
    result = sm.stats.Table2x2(t.to_numpy())
    lo, hi = result.oddsratio_confint()
    (a, b), (c, d) = t.to_numpy()
    label = exposure + (" (mga lumusong lang)" if data is waders else "")
    rows.append({"exposure": label, "a": a, "b": b, "c": c, "d": d,
                 "OR": round(result.oddsratio, 2), "CI low": round(lo, 2), "CI high": round(hi, 2)})
summary = pd.DataFrame(rows).set_index("exposure")

fig, ax = plt.subplots(figsize=(7, 4))
y = list(range(len(summary)))[::-1]  # nasa itaas ang unang exposure
for yi, (name, row) in zip(y, summary.iterrows()):
    # pula = risk factor, berde = protective, abo = tumatawid sa 1 ang CI
    colour = "#CE1126" if row["CI low"] > 1 else "#2F7D32" if row["CI high"] < 1 else "#6E7882"
    ax.plot([row["CI low"], row["CI high"]], [yi, yi], color=colour, linewidth=2.5)
    ax.plot(row["OR"], yi, "o", color=colour, markersize=8)
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xticks([0.2, 0.5, 1, 2, 5, 10], labels=["0.2", "0.5", "1", "2", "5", "10"])
ax.set_yticks(y, labels=summary.index)
ax.set_xlabel("Odds ratio (95% CI, log scale)")
ax.set_title("Mga exposure at leptospirosis, crude OR (synthetic na study)")
plt.tight_layout()
plt.show()
summary

## Hakbang 6 · Confounding: ang paglusong ba, o ang clean-up?

Tiningnan ni Doc Mon ang table at nagtanong: *"Mas madalas lumusong ang mga clean-up
volunteer, di ba? At may sariling panganib ang clean-up work — sugat, mahabang oras sa
putik. Baka bahagi ng OR ng paglusong ay galing sa clean-up?"*

Ang **confounder** ay ikatlong factor na may kaugnayan sa exposure *at* risk factor din ng
sakit nang mag-isa, kaya nahahalo ang epekto nito sa crude OR. I-check ang unang kaugnayan
sa mga **control** (ipinapakita nila kung paano magkasama ang mga exposure sa populasyon).

In [ ]:
controls = cc[cc["case"] == 0]
print("Mga control na lumusong, ayon sa clean-up work (%):")
print((pd.crosstab(controls["cleanup_work"], controls["waded_floodwater"], normalize="index")[1] * 100).round(1))
print()
print("Crude OR ng clean-up work mismo:", summary.loc["cleanup_work", "OR"])

Mas madalas lumusong ang mga clean-up worker, at may kaugnayan din sa sakit ang clean-up
work mismo. Kaya nag-**stratify** tayo: kukuwentahin nang hiwalay ang OR ng paglusong sa
mga sumama sa clean-up work at sa mga hindi. Saka pagsasamahin ang dalawang stratum gamit
ang **Mantel–Haenszel** (MH) odds ratio, isang weighted average ng mga stratum OR:

$$OR_{MH} = \frac{\sum_i a_i d_i / n_i}{\sum_i b_i c_i / n_i}$$

kung saan ang *i* ay ang stratum at ang *n_i* ay ang bilang ng tao rito.

In [ ]:
numerator = denominator = 0.0
strata = []
for work, group in cc.groupby("cleanup_work"):
    t = two_by_two(group, "waded_floodwater")
    (a, b), (c, d) = t.to_numpy()
    n = a + b + c + d
    numerator += a * d / n
    denominator += b * c / n
    strata.append(t.to_numpy())
    label = "sumama sa clean-up" if work == 1 else "hindi sa clean-up"
    print(f"{label:<18} a={a:>2} b={b:>2} c={c:>2} d={d:>3}  stratum OR = {a * d / (b * c):.2f}")

or_mh = numerator / denominator
print(f"\nCrude OR = {or_wading:.2f}   Mantel-Haenszel OR = {or_mh:.2f}")

**I-check natin** gamit ang `StratifiedTable` ng statsmodels, na nagbibigay rin ng MH confidence
interval at ng test kung mas malaki ang pagkakaiba ng mga stratum OR kaysa sa kayang
ipaliwanag ng pagkakataon.

In [ ]:
st = sm.stats.StratifiedTable(strata)
mh_low, mh_high = st.oddsratio_pooled_confint()
print(f"statsmodels MH OR = {st.oddsratio_pooled:.2f} (95% CI {mh_low:.2f}–{mh_high:.2f})")
print(f"Test kung pantay ang stratum OR (Breslow-Day): p = {st.test_equal_odds().pvalue:.2f}")
assert math.isclose(st.oddsratio_pooled, or_mh)
change = (or_wading - or_mh) / or_wading * 100
print(f"Dahil sa pag-adjust sa clean-up work, gumalaw ang OR mula {or_wading:.2f} hanggang {or_mh:.2f} ({change:.0f}% na mas maliit)")

**Ano ang nagbago at bakit.** Pinaghalo ng crude OR (5.57) ang dalawang bagay: ang risk ng
paglusong mismo at ang dagdag na risk ng clean-up work, dahil mas madalas lumusong ang mga
clean-up worker. Sa loob ng bawat stratum, hindi gumagalaw ang clean-up factor, at bumababa
sa **4.28** ang pinagsamang MH OR. Malakas pa ring kaugnay ng leptospirosis ang paglusong
— hindi nawala ang association — pero bahagi ng crude na numero ay galing sa clean-up work.
Magkaiba ang itsura ng mga stratum OR (5.56 at 2.88), pero sa mga numerong ito, hindi
masabi ng test kung higit pa ito sa nagkataon (p ≈ 0.24), kaya makatwiran ang iisang MH
summary. Maraming magkakabuhol na exposure at sampling error ang totoong mga study, kaya
bihirang crude OR na ang buong katotohanan.

**Ang rekomendasyon ni Maya sa City Health Officer:**

> Malakas na kaugnay ng leptospirosis ang paglusong sa baha pagkatapos ng Amihan, kahit
> isinaalang-alang na ang clean-up work (adjusted OR na mga 4), at mas mababa nang malaki
> ang odds sa mga nakabota sa mga lumusong. Bago ang susunod na baha, dapat magbigay ang
> lungsod ng bota at wound care sa mga clean-up crew at sa mga residenteng kailangang
> lumusong, at sabihan ang sinumang lagnatin pagkatapos madikit sa baha na magpakonsulta
> agad. Kaugnay ng mas mababang odds ng sakit ang doxycycline prophylaxis sa study na ito,
> kaya dapat ituloy ang pamamahagi nito sa mga high-exposure group, ayon sa kasalukuyang
> gabay ng DOH.

## Ikaw naman

1. **Sugat sa balat.** Na-confound din ba ng clean-up work ang OR ng `skin_wound`? *Hint:
   kopyahin ang Mantel–Haenszel cell at palitan ang `"waded_floodwater"` ng `"skin_wound"`.*
2. **Talaga bang mas mataas ang risk ng mga lalaki?** 2.82 ang crude OR para sa `male`.
   I-stratify ito ayon sa `waded_floodwater` at i-compute ang MH OR. *Hint: mas madalas
   lumusong ang mga lalaki; asahang liliit ang OR.*
3. **Ulitin nang sadya ang pagkakamali ni Tala.** Baligtarin lang ang rows ng tamang table
   gamit ang `table.iloc[::-1]` at i-compute ang $ad/bc$. Ipakitang `1 / or_wading` ang
   sagot. *Hint: kaibigan mo ang `math.isclose(...)`.*